# Practical Tips

## Activation Functions

In [ ]:
from keras.layers import LeakyReLU

model = Sequential()
model.add(Dense(64, input_shape=(784,)))
model.add(LeakyReLU(alpha=0.1))
model.add(Dense(64))
model.add(LeakyReLU(alpha=0.1))


In [ ]:
from keras.layers import ELU

model = Sequential([
    Dense(64, input_shape=(784,)),
    ELU(alpha=1.0),
    Dense(64, activation='elu'),
    Dense(10, activation='softmax')])

In [ ]:
model = Sequential([
    Dense(64, input_shape=(784, ), activation='selu', kernel_initializer='lecun_normal'),
    Dense(64, activation='selu', kernel_initializer='lecun_normal'),
    Dense(10, activation='softmax')])

In [ ]:
from keras.layers import Activation
from keras.activations import elu

def selu(x, alpha=1.67326, scale=1.0507):
    return scale * elu(x, alpha)

custom_selu =lambda x: selu(x, alpha=1, scale=1)

model = Sequential([
    Dense(64, input_shape=(784, ), kernel_initializer='lecun_normal'),
    Activation(custom_selu),
    Dense(64, kernel_initializer='lecun_normal'),
    Activation(custom_selu),
    Dense(10, activation='softmax')])


## Batch Normalization


In [ ]:
from keras.layers import BatchNormalization

model = Sequential([
    BatchNormalization(),
    Dense(64, input_shape=(784, ), activation='relu'),
    BatchNormalization(),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(10, activation='softmax')])

## Optimizers

+ SGD 

$W_{t+1} = W_t - \alpha \cdot g$

+ SGD with momentum

$v_{t+1} = \mu \cdot v_t - \alpha \cdot g$

$W_{t+1} = W_t + v_{t+1}$

+ Nesterov Accelerated Gradient (NAG)

$v_{t+1} = \mu \cdot v_t - \alpha \cdot \nabla f(W_t + \mu \cdot v_t)$

$W_{t+1} = W_t + v_{t+1}$


In [ ]:
from keras.optimizers import SGD

# SGD 
optimizer = SGD(learning_rate=0.01)

# SGD with momentum
optimizer = SGD(learning_rate=0.01, momentum=0.9)

# Nesterov Accelerated Gradient (NAG)
optimizer = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

+ Adagrad

$v_{t+1} = v_t + g^2$

$W_{t+1} = W_t - \frac{\alpha}{\sqrt{v_{t+1}} + \epsilon} \cdot g$

+ RMSprop

$v_{t+1} = \beta \cdot v_t + (1 - \beta) \cdot g^2$

$W_{t+1} = W_t - \alpha \cdot \frac{g}{\sqrt{v_{t+1}} + \epsilon}$



In [ ]:
from keras.optimizers import Adagrad, RMSprop

optimizer = Adagrad(learning_rate=0.01)

optimizer = RMSprop(learning_rate=0.001)



## Losses

In [ ]:
from keras.losses import CategoricalCrossentropy, SparseCategoricalCrossentropy

y1 = [[0, 1, 0], [0, 0, 1]]
y2 = [1, 2]
y_pred = [[0.05, 0.95, 0], [0.1, 0.8, 0.1]]

cat_entropy = CategoricalCrossentropy()
sparse_cat_entropy = SparseCategoricalCrossentropy()

print(cat_entropy (y1, y_pred).numpy())
print(sparse_cat_entropy (y2, y_pred).numpy())

1.1769392
1.1769392


In [ ]:
y3 = [[0, 1, 1], [0, 0, 1]]
y_pred = [[0.05, 0.95, 0.9], [0.1, 0.8, 0.1]]

print(cat_entropy (y3, y_pred).numpy())
# print(sparse_cat_entropy (y1, y_pred).numpy())

1.8714733


## Regularization


### L1 and L2 regularization

In [ ]:

from keras.regularizers import l1, l2, l1_l2

model = Sequential([
    Dense(64, input_shape=(784, ), activation='relu', kernel_regularizer=l1(0.01)),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(64, activation='relu', kernel_regularizer=l1_l2(0.01, 0.01)),
    Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from functools import partial
from keras.layers import Dense, Input

RegularizedDense = partial(Dense, activation="elu",
                        kernel_initializer="he_normal", kernel_regularizer= l2(0.01))

model = Sequential([
                    Input (shape=(784,)),
                    RegularizedDense(300),
                    RegularizedDense(100),
                    RegularizedDense(10, activation="softmax",
                    kernel_initializer="glorot_uniform")
                    ])

In [ ]:
model = Sequential()
model.add(Dense(64, input_shape=(784, ), activation='relu'))
for i in range(4,1,-1):
    model.add(Dense(i * 64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dense(10, activation='softmax'))

### Dropout

In [ ]:
from keras.layers import Dropout

model = Sequential()
model.add(Dense(64, input_shape=(784, ), activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(10, activation='softmax'))

### Early Stopping

In [ ]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=3, min_delta=0.001, restore_best_weights=True)

model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[early_stopping])

## Model Checkpointing

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint('my_keras_model.h5', monitor='val_acc', save_best_only=True)
early_stopping = EarlyStopping(monitor='val_acc', patience=10, restore_best_weights=True)

model.fit(X_train, y_train, epochs=100, validation_data=(X_valid, y_valid), 
            callbacks=[checkpoint, early_stopping])
